# 원본 데이터 약물 종류 갯수 파악

In [5]:
import os
import json
import cv2
import glob
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. 경로 및 환경 설정
# ==========================================
base_path = r"C:\Users\home\Desktop\AI_study\data"
save_root = r"C:\Users\home\Desktop\AI_study\processed_crops"
error_root = r"C:\Users\home\Desktop\AI_study\error_samples"

pairs = [
    (os.path.join(base_path, "train_images"), os.path.join(base_path, "train_annotations"), "OLD"),
    (os.path.join(base_path, "add_images"), os.path.join(base_path, "add_annotations"), "NEW")
]

# ==========================================
# 2. 기능 함수
# ==========================================
def get_target_drug_list():
    """
    [수정됨] 폴더 깊이 상관없이 모든 하위 디렉토리를 뒤져서
    'K-'로 시작하는 약물 코드를 찾아냅니다.
    """
    print("📋 [Step 0] 족보(Target List) 추출 중 (Deep Scan)...")
    target_drugs = set()
    train_lbl_dir = os.path.join(base_path, "train_annotations")
    
    if not os.path.exists(train_lbl_dir):
        print("❌ 경로 에러: train_annotations 폴더가 없습니다.")
        return set()

    # os.walk로 모든 구석구석 탐색
    for root, dirs, files in os.walk(train_lbl_dir):
        for d in dirs:
            # 약물 코드는 K-로 시작한다고 가정 (안전장치: 길이 5 이상)
            if d.startswith("K-") and len(d) > 5:
                target_drugs.add(d)
            
    print(f"✅ 족보 확보 완료: 총 {len(target_drugs)} 종의 약물을 학습합니다.")
    return target_drugs

def preprocess_pipeline():
    target_list = get_target_drug_list()
    if not target_list: return

    os.makedirs(save_root, exist_ok=True)
    os.makedirs(error_root, exist_ok=True)

    stats = {
        "success": 0, "idx_skip": 0, "no_label": 0, 
        "not_target": 0, "no_json": 0, "bbox_err": 0
    }

    print("\n🚀 [전처리 파이프라인 가동] 구조 불문, K-코드 싹 다 찾기")

    for img_dir, label_dir, source_tag in pairs:
        if not os.path.exists(img_dir): continue

        img_files = glob.glob(os.path.join(img_dir, "*.*"))
        label_map = {f.replace('_json', ''): f for f in os.listdir(label_dir)}

        for img_path in tqdm(img_files, desc=f"Processing {source_tag}"):
            filename = os.path.basename(img_path)
            
            # [Step 1.5] Index 제거
            if 'index' in filename.lower():
                stats["idx_skip"] += 1
                continue

            ext = os.path.splitext(filename)[1].lower()
            if ext not in ['.jpg', '.png', '.jpeg']: continue
            img_key = os.path.splitext(filename)[0]

            # [Step 1] 매핑
            matched_folder = label_map.get(img_key)
            if not matched_folder:
                for k, v in label_map.items():
                    if img_key.startswith(k):
                        matched_folder = v
                        break
            
            if not matched_folder:
                stats["no_label"] += 1
                continue

            # 이미지 로드
            img = cv2.imread(img_path)
            if img is None: continue
            h_img, w_img, _ = img.shape

            # [Step 2] 약물 탐색 (구조 유연화)
            combo_path = os.path.join(label_dir, matched_folder)
            
            # 이 폴더 안에 있는 'K-' 폴더들을 찾는다.
            # 만약 combo_path 자체가 'K-' 폴더일 수도 있고, 그 안에 있을 수도 있음.
            found_drugs = []
            
            # 1. 하위 폴더 중에 K- 코드가 있는지 검색
            sub_dirs = [d for d in os.listdir(combo_path) if os.path.isdir(os.path.join(combo_path, d))]
            k_sub_dirs = [d for d in sub_dirs if d.startswith("K-")]
            
            if k_sub_dirs:
                # 하위 폴더가 약물인 경우 (3단 구조)
                for d in k_sub_dirs:
                    found_drugs.append((d, os.path.join(combo_path, d)))
            else:
                # 하위 폴더가 없으면? 혹시 매칭된 폴더 이름 자체가 K- 코드인가? (2단 구조)
                # 예: matched_folder가 "K-000250_json"이 아니라 "K-000250"인 경우 등
                folder_name = matched_folder.replace('_json', '')
                if folder_name.startswith("K-"):
                    found_drugs.append((folder_name, combo_path))

            # 약물을 하나도 못 찾았다면?
            if not found_drugs:
                # 데이터 구조가 이상하거나 빈 폴더임
                continue

            for drug_code, drug_path in found_drugs:
                # 🛑 타겟 필터링
                if drug_code not in target_list:
                    stats["not_target"] += 1
                    continue

                # [Step 3] JSON & 크롭
                json_files = glob.glob(os.path.join(drug_path, "*.json"))
                if not json_files:
                    stats["no_json"] += 1
                    continue
                
                try:
                    with open(json_files[0], 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        bbox = None
                        if 'annotations' in data and len(data['annotations']) > 0:
                            bbox = data['annotations'][0].get('bbox')
                        
                        if not bbox:
                            stats["no_json"] += 1
                            continue

                        x, y, w, h = map(int, bbox)
                        if w < 10 or h < 10: 
                            stats["bbox_err"] += 1; continue
                        
                        x = max(0, x); y = max(0, y)
                        w = min(w, w_img - x); h = min(h, h_img - y)
                        if w <= 0 or h <= 0: 
                            stats["bbox_err"] += 1; continue

                        crop = img[y:y+h, x:x+w]
                        
                        save_dir = os.path.join(save_root, drug_code)
                        os.makedirs(save_dir, exist_ok=True)
                        save_name = f"{source_tag}_{img_key}_{drug_code}.jpg"
                        cv2.imwrite(os.path.join(save_dir, save_name), crop)
                        stats["success"] += 1
                except:
                    stats["bbox_err"] += 1
                    continue

    print("\n" + "="*50)
    print("📊 최종 결과 리포트 (수정본)")
    print("="*50)
    print(f"✅ [성공] 저장된 크롭 이미지   : {stats['success']} 장")
    print(f"🚫 [스킵] 족보에 없는 약(Target X): {stats['not_target']} 건")
    print("-" * 50)
    
    # 약물별 개수 체크 (76종 나오는지 확인)
    saved_drugs = os.listdir(save_root)
    print(f"📂 확보된 약물 종류: {len(saved_drugs)} 종")
    if len(saved_drugs) < 70:
        print("⚠️ 경고: 여전히 76종보다 적습니다. 데이터 자체 문제일 수 있습니다.")
    else:
        print("🎉 성공: 누락되었던 약물들을 모두 찾았습니다!")

preprocess_pipeline()

📋 [Step 0] 족보(Target List) 추출 중 (Deep Scan)...
✅ 족보 확보 완료: 총 170 종의 약물을 학습합니다.

🚀 [전처리 파이프라인 가동] 구조 불문, K-코드 싹 다 찾기


Processing OLD: 100%|██████████| 232/232 [00:09<00:00, 24.99it/s]
Processing NEW: 0it [00:00, ?it/s]


📊 최종 결과 리포트 (수정본)
✅ [성공] 저장된 크롭 이미지   : 769 장
🚫 [스킵] 족보에 없는 약(Target X): 0 건
--------------------------------------------------
📂 확보된 약물 종류: 56 종
⚠️ 경고: 여전히 76종보다 적습니다. 데이터 자체 문제일 수 있습니다.


# 원본 데이터 + 새 데이터(허브AI 1~8) 이미지 갯수 파악(크롭 전)

In [9]:
import os
import pandas as pd
from tqdm import tqdm

# ==========================================
# 사용자 설정
# ==========================================
base_path = r"C:\Users\home\Desktop\AI_study\data"
save_csv_path = r"C:\Users\home\Desktop\AI_study\final_counts.csv" # 엑셀 확인용

def count_and_show_all():
    print("🚀 [전수 조사] 모든 데이터(1~8) 뒤져서 카운트 중...")
    
    stats = {}

    # 1. 모든 폴더 탐색
    for root, dirs, files in os.walk(base_path):
        json_files = [f for f in files if f.lower().endswith('.json')]
        if not json_files: continue

        # 2. OLD vs NEW 판별
        is_old = 'train_' in root
        is_new = 'add' in root
        if not is_old and not is_new: continue

        # 3. 약물 코드 추출
        folder_name = os.path.basename(root)
        drug_code = None
        
        if folder_name.startswith("K-") and len(folder_name) > 5:
            drug_code = folder_name
        else:
            parent = os.path.basename(os.path.dirname(root))
            if parent.startswith("K-") and len(parent) > 5:
                drug_code = parent
        
        if not drug_code: continue
        if drug_code.count('-') > 1: continue # 껍데기 폴더 제외

        # 4. 집계
        if drug_code not in stats:
            stats[drug_code] = {'OLD': 0, 'NEW': 0}
        
        if is_old:
            stats[drug_code]['OLD'] += len(json_files)
        elif is_new:
            stats[drug_code]['NEW'] += len(json_files)

    # 5. OLD=0 제거 (필터링)
    valid_stats = {k: v for k, v in stats.items() if v['OLD'] > 0}
    removed_count = len(stats) - len(valid_stats)

    # DataFrame 생성
    df = pd.DataFrame.from_dict(valid_stats, orient='index').reset_index()
    df.columns = ['Drug_Code', 'OLD_Count', 'NEW_Count']
    df['Total'] = df['OLD_Count'] + df['NEW_Count']
    
    # 정렬
    df = df.sort_values(by='Total', ascending=False).reset_index(drop=True)

    # ==========================================
    # 🔥 [강제 출력] to_string()으로 생략 방지
    # ==========================================
    print("\n" + "="*60)
    print(f"✅ 최종 집계 완료: 총 {len(df)} 종 (OLD 데이터가 없는 {removed_count}종 제외됨)")
    print("="*60)
    
    # 여기서 화면에 다 뿌립니다.
    print(df.to_string()) 
    
    print("-" * 60)
    
    # 엑셀(CSV) 저장
    df.to_csv(save_csv_path, index=False, encoding='utf-8-sig')
    print(f"💾 파일로도 저장했습니다: {save_csv_path}")
    print("   (메모장이나 엑셀로 열어서 확인해보세요!)")

# 실행
count_and_show_all()

🚀 [전수 조사] 모든 데이터(1~8) 뒤져서 카운트 중...

✅ 최종 집계 완료: 총 56 종 (OLD 데이터가 없는 62종 제외됨)
   Drug_Code  OLD_Count  NEW_Count  Total
0   K-016548         18       1098   1116
1   K-016551          3       1095   1098
2   K-016232         21        942    963
3   K-016262         23        708    731
4   K-035206         37        684    721
5   K-029667         18        699    717
6   K-020238         20        696    716
7   K-036637         19        693    712
8   K-003483         45        624    669
9   K-022347          9        558    567
10  K-025367         12        549    561
11  K-025469         12        546    558
12  K-020877         12        543    555
13  K-028763          9        540    549
14  K-027733         12        534    546
15  K-003743          3        540    543
16  K-030308          9        531    540
17  K-019861          6        531    537
18  K-034597          3        531    534
19  K-031885          3        528    531
20  K-027777          3        521    524

# 크롭 데이터 확인

In [11]:
import os
import json
import cv2
import glob

# ==========================================
# 사용자 설정
# ==========================================
base_path = r"C:\Users\home\Desktop\AI_study\data"
save_debug_path = r"C:\Users\home\Desktop\AI_study\debug_visualize"

# 문제가 된 그 약물 코드
target_drug_code = "K-002483" 

def debug_visualize_fixed():
    os.makedirs(save_debug_path, exist_ok=True)
    print(f"🕵️‍♂️ 범인 색출 재시도: {target_drug_code} 추적 중...")

    # 1. 라벨 폴더 설정 (train, add1~8 모두 포함해서 찾기 위해 상위 경로 사용)
    # 하지만 일단 확실한 OLD 데이터부터 뒤집니다.
    lbl_root = os.path.join(base_path, "train_annotations")
    img_root = os.path.join(base_path, "train_images")

    count = 0

    # 2. 라벨 폴더 뒤지기
    for root, dirs, files in os.walk(lbl_root):
        # 현재 폴더가 타겟 약물 폴더인지 확인
        # (폴더명 자체가 K-002483 이어야 함)
        if os.path.basename(root) == target_drug_code:
            
            json_files = [f for f in files if f.endswith(".json")]
            
            for j_file in json_files:
                try:
                    # JSON 읽기
                    json_path = os.path.join(root, j_file)
                    with open(json_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    if not data['annotations']: continue
                    
                    bbox = data['annotations'][0]['bbox']
                    x, y, w, h = map(int, bbox)

                    # 3. 대응되는 이미지 찾기 (이미지 폴더에서)
                    # JSON 파일명 = 이미지 파일명 (확장자만 다름)
                    img_name_base = os.path.splitext(j_file)[0]
                    
                    # jpg, png, jpeg 다 찾아봄
                    img_candidates = glob.glob(os.path.join(img_root, img_name_base + ".*"))
                    
                    if not img_candidates:
                        # OLD에 없으면 NEW(add1_images)에서도 찾아봐야 함 (혹시 모르니)
                        img_candidates = glob.glob(os.path.join(base_path, "add1_images", img_name_base + ".*"))
                    
                    if not img_candidates:
                        print(f"⚠️ 이미지 파일 없음: {img_name_base}")
                        continue
                    
                    img_path = img_candidates[0]
                    
                    # 4. 이미지 로드 및 그리기
                    img = cv2.imread(img_path)
                    if img is None: continue

                    # 빨간 박스 (원본 좌표)
                    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 5)
                    
                    # 텍스트
                    cv2.putText(img, f"{target_drug_code}", (x, y-10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

                    # 저장
                    save_name = f"CHECK_{target_drug_code}_{img_name_base}.jpg"
                    cv2.imwrite(os.path.join(save_debug_path, save_name), img)
                    print(f"📸 캡처 완료: {save_name}")
                    
                    count += 1
                    if count >= 20: # 20장만 뽑고 종료
                        print("🛑 20장 확인 완료. debug_visualize 폴더를 확인하세요.")
                        return

                except Exception as e:
                    print(f"에러: {e}")
                    continue
    
    if count == 0:
        print("❌ 여전히 파일을 못 찾았습니다. 경로 설정을 다시 확인해야 합니다.")
        print(f"검색한 라벨 경로: {lbl_root}")
    else:
        print(f"📂 확인 위치: {save_debug_path}")

# 실행
debug_visualize_fixed()

🕵️‍♂️ 범인 색출 재시도: K-002483 추적 중...
📸 캡처 완료: CHECK_K-002483_K-002483-003743-012778-013395_0_2_0_2_70_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-003743-012778-013395_0_2_0_2_75_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-003743-012778-013395_0_2_0_2_90_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-012081-012778-025438_0_2_0_2_70_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-012081-012778-025438_0_2_0_2_75_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-012081-012778-025438_0_2_0_2_90_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-019552-022362-025438_0_2_0_2_70_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-019552-022362-025438_0_2_0_2_75_000_200.jpg
📸 캡처 완료: CHECK_K-002483_K-002483-019552-022362-025438_0_2_0_2_90_000_200.jpg
📂 확인 위치: C:\Users\home\Desktop\AI_study\debug_visualize


# 원본 데이터 1.5배로 크롭

In [12]:
import os
import json
import cv2
import glob
import numpy as np
from tqdm import tqdm

# ==========================================
# 🔒 사용자 지정 경로 (딱 여기만 봅니다)
# ==========================================
lbl_root = r"C:\Users\home\Desktop\AI_study\data\train_annotations"
img_root = r"C:\Users\home\Desktop\AI_study\data\train_images"

# 결과 저장소 (구분을 위해 이름 변경)
save_root = r"C:\Users\home\Desktop\AI_study\processed_crops_OLD_ONLY"
SCALE_FACTOR = 1.5  # 1.5배 확장

def process_old_data_only():
    os.makedirs(save_root, exist_ok=True)
    print(f"🔒 [검증 모드] 오직 OLD 데이터만 처리합니다.")
    print(f"   - 라벨: {lbl_root}")
    print(f"   - 이미지: {img_root}")
    
    stats = {"success": 0, "skip": 0, "error": 0}

    # 1. 라벨 폴더 탐색 (os.walk로 하위 폴더까지)
    for root, dirs, files in os.walk(lbl_root):
        
        # 현재 폴더가 약물 코드인지 확인 (K-...)
        # OLD 데이터 구조: train_annotations / 조합명 / 약물명(K-...) / json
        folder_name = os.path.basename(root)
        
        # 약물 코드가 아니면 스킵 (안전장치)
        if not (folder_name.startswith("K-") and len(folder_name) > 5):
            continue
            
        drug_code = folder_name
        
        # 2. JSON 파일 처리
        json_files = [f for f in files if f.endswith(".json")]
        
        for j_file in tqdm(json_files, desc=f"Processing {drug_code}", leave=False):
            try:
                # JSON 로드
                json_path = os.path.join(root, j_file)
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                if not data['annotations']: continue
                bbox = data['annotations'][0]['bbox']
                x, y, w, h = map(int, bbox)

                # 3. 이미지 매칭 (train_images 폴더에서 찾기)
                # 파일명 규칙: JSON이름.json -> 이미지는 같은 이름.jpg/png
                base_name = os.path.splitext(j_file)[0]
                
                # 이미지 폴더 전체 뒤지면 느리니까, 경로 추적 시도 X -> glob으로 이름 매칭
                # train_images 폴더 안에 이미지가 평평하게(flat) 있는지, 구조가 있는지 모름
                # 안전하게 recursive glob으로 찾음 (처음에만 좀 걸림)
                img_candidates = glob.glob(os.path.join(img_root, "**", base_name + ".*"), recursive=True)
                
                if not img_candidates:
                    stats["skip"] += 1
                    continue
                
                img_path = img_candidates[0]
                img = cv2.imread(img_path)
                if img is None: continue
                
                img_h, img_w, _ = img.shape

                # ==========================================
                # 🔥 [수정된 로직] 1.5배 확장 + 마이너스 좌표 방지
                # ==========================================
                center_x, center_y = x + w/2, y + h/2
                new_w, new_h = w * SCALE_FACTOR, h * SCALE_FACTOR
                
                new_x = center_x - new_w/2
                new_y = center_y - new_h/2
                
                # ✅ 핵심: max(0, ...)으로 음수 좌표를 0으로 강제 변환
                # 이걸 안 하면 이미지가 뒤틀리거나 반대편이 찍힘
                x1 = int(max(0, new_x))
                y1 = int(max(0, new_y))
                x2 = int(min(img_w, new_x + new_w))
                y2 = int(min(img_h, new_y + new_h))
                
                if x2 <= x1 or y2 <= y1: 
                    stats["error"] += 1; continue

                crop = img[y1:y2, x1:x2]
                
                # 4. 저장
                save_dir = os.path.join(save_root, drug_code)
                os.makedirs(save_dir, exist_ok=True)
                
                save_name = f"OLD_{base_name}_{drug_code}.jpg"
                cv2.imwrite(os.path.join(save_dir, save_name), crop)
                stats["success"] += 1

            except Exception as e:
                stats["error"] += 1
                continue

    print("=" * 50)
    print(f"✅ OLD 데이터 검증 완료")
    print(f"   - 성공: {stats['success']}장")
    print(f"   - 실패/스킵: {stats['skip'] + stats['error']}장")
    print(f"📂 저장 위치: {save_root}")
    print("=" * 50)

# 실행
process_old_data_only()

🔒 [검증 모드] 오직 OLD 데이터만 처리합니다.
   - 라벨: C:\Users\home\Desktop\AI_study\data\train_annotations
   - 이미지: C:\Users\home\Desktop\AI_study\data\train_images


✅ OLD 데이터 검증 완료
   - 성공: 762장
   - 실패/스킵: 1장
📂 저장 위치: C:\Users\home\Desktop\AI_study\processed_crops_OLD_ONLY


# 원본 데이터 1.0배로 크롭

In [2]:
import os
import json
import cv2
import glob
import numpy as np
from tqdm import tqdm

# ==========================================
# 🔒 사용자 지정 경로 (딱 여기만 봅니다)
# ==========================================
lbl_root = r"C:\Users\home\Desktop\AI_study\data\train_annotations"
img_root = r"C:\Users\home\Desktop\AI_study\data\train_images"

# 결과 저장소 (구분을 위해 이름 변경)
save_root = r"C:\Users\home\Desktop\AI_study\processed_crops_OLD_ONLY1.2"
SCALE_FACTOR = 1.2  # 1.5배 확장

def process_old_data_only():
    os.makedirs(save_root, exist_ok=True)
    print(f"🔒 [검증 모드] 오직 OLD 데이터만 처리합니다.")
    print(f"   - 라벨: {lbl_root}")
    print(f"   - 이미지: {img_root}")
    
    stats = {"success": 0, "skip": 0, "error": 0}

    # 1. 라벨 폴더 탐색 (os.walk로 하위 폴더까지)
    for root, dirs, files in os.walk(lbl_root):
        
        # 현재 폴더가 약물 코드인지 확인 (K-...)
        # OLD 데이터 구조: train_annotations / 조합명 / 약물명(K-...) / json
        folder_name = os.path.basename(root)
        
        # 약물 코드가 아니면 스킵 (안전장치)
        if not (folder_name.startswith("K-") and len(folder_name) > 5):
            continue
            
        drug_code = folder_name
        
        # 2. JSON 파일 처리
        json_files = [f for f in files if f.endswith(".json")]
        
        for j_file in tqdm(json_files, desc=f"Processing {drug_code}", leave=False):
            try:
                # JSON 로드
                json_path = os.path.join(root, j_file)
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                if not data['annotations']: continue
                bbox = data['annotations'][0]['bbox']
                x, y, w, h = map(int, bbox)

                # 3. 이미지 매칭 (train_images 폴더에서 찾기)
                # 파일명 규칙: JSON이름.json -> 이미지는 같은 이름.jpg/png
                base_name = os.path.splitext(j_file)[0]
                
                # 이미지 폴더 전체 뒤지면 느리니까, 경로 추적 시도 X -> glob으로 이름 매칭
                # train_images 폴더 안에 이미지가 평평하게(flat) 있는지, 구조가 있는지 모름
                # 안전하게 recursive glob으로 찾음 (처음에만 좀 걸림)
                img_candidates = glob.glob(os.path.join(img_root, "**", base_name + ".*"), recursive=True)
                
                if not img_candidates:
                    stats["skip"] += 1
                    continue
                
                img_path = img_candidates[0]
                img = cv2.imread(img_path)
                if img is None: continue
                
                img_h, img_w, _ = img.shape

                # ==========================================
                # 🔥 [수정된 로직] 1.5배 확장 + 마이너스 좌표 방지
                # ==========================================
                center_x, center_y = x + w/2, y + h/2
                new_w, new_h = w * SCALE_FACTOR, h * SCALE_FACTOR
                
                new_x = center_x - new_w/2
                new_y = center_y - new_h/2
                
                # ✅ 핵심: max(0, ...)으로 음수 좌표를 0으로 강제 변환
                # 이걸 안 하면 이미지가 뒤틀리거나 반대편이 찍힘
                x1 = int(max(0, new_x))
                y1 = int(max(0, new_y))
                x2 = int(min(img_w, new_x + new_w))
                y2 = int(min(img_h, new_y + new_h))
                
                if x2 <= x1 or y2 <= y1: 
                    stats["error"] += 1; continue

                crop = img[y1:y2, x1:x2]
                
                # 4. 저장
                save_dir = os.path.join(save_root, drug_code)
                os.makedirs(save_dir, exist_ok=True)
                
                save_name = f"OLD_{base_name}_{drug_code}.jpg"
                cv2.imwrite(os.path.join(save_dir, save_name), crop)
                stats["success"] += 1

            except Exception as e:
                stats["error"] += 1
                continue

    print("=" * 50)
    print(f"✅ OLD 데이터 검증 완료")
    print(f"   - 성공: {stats['success']}장")
    print(f"   - 실패/스킵: {stats['skip'] + stats['error']}장")
    print(f"📂 저장 위치: {save_root}")
    print("=" * 50)

# 실행
process_old_data_only()

🔒 [검증 모드] 오직 OLD 데이터만 처리합니다.
   - 라벨: C:\Users\home\Desktop\AI_study\data\train_annotations
   - 이미지: C:\Users\home\Desktop\AI_study\data\train_images


✅ OLD 데이터 검증 완료
   - 성공: 762장
   - 실패/스킵: 1장
📂 저장 위치: C:\Users\home\Desktop\AI_study\processed_crops_OLD_ONLY1.2


# 새 데이터 1.0배로 크롭

In [15]:
import os
import json
import cv2
import glob
import numpy as np
from tqdm import tqdm

# ==========================================
# 📂 경로 설정
# ==========================================
old_lbl_root = r"C:\Users\home\Desktop\AI_study\data\train_annotations" # 족보용
new_lbl_root = r"C:\Users\home\Desktop\AI_study\data\add_annotations"  # 라벨
new_img_root = r"C:\Users\home\Desktop\AI_study\data\add_images"      # 이미지

save_root = r"C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY"
SCALE_FACTOR = 1.0 # 1.0배 (딱 맞게) 혹은 1.2배 등 원하시는 대로 수정!

def get_allowed_drug_list():
    """ 족보(56종) 추출 """
    print("📋 족보(56종) 확인 중...")
    allowed_drugs = set()
    if os.path.exists(old_lbl_root):
        for root, dirs, files in os.walk(old_lbl_root):
            for d in dirs:
                if d.startswith("K-") and len(d) > 5 and d.count('-') == 1:
                    allowed_drugs.add(d)
    return allowed_drugs

def build_image_map(img_root):
    """ 🔥 [핵심] 이미지 파일 경로를 미리 다 찾아서 '주소록'을 만듭니다. """
    print("🗺️ 이미지 위치 주소록(Indexing) 만드는 중... (잠시만 기다려주세요)")
    image_map = {}
    
    # os.walk로 딱 한 번만 전체 스캔
    for root, dirs, files in os.walk(img_root):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                # 키: 파일명(확장자 제외), 값: 전체경로
                base_name = os.path.splitext(file)[0]
                image_map[base_name] = os.path.join(root, file)
                
    print(f"✅ 주소록 완성! 총 {len(image_map)}개의 이미지 위치를 파악했습니다.")
    return image_map

def process_new_data_fast():
    # 1. 준비 단계
    target_list = get_allowed_drug_list()
    if not target_list: return
    
    # 여기서 이미지 주소록을 먼저 만듭니다 (시간 단축의 핵심)
    img_map = build_image_map(new_img_root)

    os.makedirs(save_root, exist_ok=True)
    print(f"🚀 [초고속 모드] NEW 데이터 크롭 시작 (대상: 56종)")
    
    stats = {"success": 0, "skip_filter": 0, "skip_no_img": 0}

    # 2. 라벨 탐색
    for root, dirs, files in os.walk(new_lbl_root):
        folder_name = os.path.basename(root)
        
        # 약물 코드 추출
        drug_code = None
        if folder_name.startswith("K-") and len(folder_name) > 5:
            drug_code = folder_name
        
        if not drug_code: continue

        # 족보 필터링
        if drug_code not in target_list:
            stats["skip_filter"] += len([f for f in files if f.endswith(".json")])
            continue 

        # 3. 고속 처리 시작
        json_files = [f for f in files if f.endswith(".json")]
        
        # tqdm으로 진행률 표시
        for j_file in tqdm(json_files, desc=f"{drug_code}", leave=False):
            try:
                # JSON 로드
                json_path = os.path.join(root, j_file)
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                if not data['annotations']: continue
                bbox = data['annotations'][0]['bbox']
                x, y, w, h = map(int, bbox)

                # 🔥 [핵심] 주소록에서 바로 조회 (검색 시간 0초)
                base_name = os.path.splitext(j_file)[0]
                img_path = img_map.get(base_name) # 여기서 바로 찾음!
                
                if not img_path:
                    stats["skip_no_img"] += 1; continue
                
                img = cv2.imread(img_path)
                if img is None: continue
                img_h, img_w, _ = img.shape

                # 좌표 계산 (안전 크롭)
                center_x, center_y = x + w/2, y + h/2
                new_w, new_h = w * SCALE_FACTOR, h * SCALE_FACTOR
                new_x, new_y = center_x - new_w/2, center_y - new_h/2
                
                x1 = int(max(0, new_x))
                y1 = int(max(0, new_y))
                x2 = int(min(img_w, new_x + new_w))
                y2 = int(min(img_h, new_y + new_h))
                
                if x2 <= x1 or y2 <= y1: continue

                crop = img[y1:y2, x1:x2]
                
                # 저장
                save_dir = os.path.join(save_root, drug_code)
                os.makedirs(save_dir, exist_ok=True)
                save_name = f"NEW_{base_name}_{drug_code}.jpg"
                cv2.imwrite(os.path.join(save_dir, save_name), crop)
                stats["success"] += 1

            except:
                continue

    print("=" * 50)
    print(f"✅ 초고속 작업 완료")
    print(f"   - 성공: {stats['success']}장")
    print(f"   - 필터링(버림): {stats['skip_filter']}장")
    print(f"   - 이미지 못찾음: {stats['skip_no_img']}장")
    print(f"📂 저장 위치: {save_root}")
    print("=" * 50)

# 실행
process_new_data_fast()

📋 족보(56종) 확인 중...
🗺️ 이미지 위치 주소록(Indexing) 만드는 중... (잠시만 기다려주세요)
✅ 주소록 완성! 총 16012개의 이미지 위치를 파악했습니다.
🚀 [초고속 모드] NEW 데이터 크롭 시작 (대상: 56종)


✅ 초고속 작업 완료
   - 성공: 23083장
   - 필터링(버림): 22921장
   - 이미지 못찾음: 17장
📂 저장 위치: C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY


# 새 데이터 1.2배로 크롭

In [1]:
import os
import json
import cv2
import glob
import numpy as np
from tqdm import tqdm

# ==========================================
# 📂 경로 설정
# ==========================================
old_lbl_root = r"C:\Users\home\Desktop\AI_study\data\train_annotations" # 족보용
new_lbl_root = r"C:\Users\home\Desktop\AI_study\data\add_annotations"  # 라벨
new_img_root = r"C:\Users\home\Desktop\AI_study\data\add_images"      # 이미지

save_root = r"C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY1.2"
SCALE_FACTOR = 1.2 # 1.0배 (딱 맞게) 혹은 1.2배 등 원하시는 대로 수정!

def get_allowed_drug_list():
    """ 족보(56종) 추출 """
    print("📋 족보(56종) 확인 중...")
    allowed_drugs = set()
    if os.path.exists(old_lbl_root):
        for root, dirs, files in os.walk(old_lbl_root):
            for d in dirs:
                if d.startswith("K-") and len(d) > 5 and d.count('-') == 1:
                    allowed_drugs.add(d)
    return allowed_drugs

def build_image_map(img_root):
    """ 🔥 [핵심] 이미지 파일 경로를 미리 다 찾아서 '주소록'을 만듭니다. """
    print("🗺️ 이미지 위치 주소록(Indexing) 만드는 중... (잠시만 기다려주세요)")
    image_map = {}
    
    # os.walk로 딱 한 번만 전체 스캔
    for root, dirs, files in os.walk(img_root):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                # 키: 파일명(확장자 제외), 값: 전체경로
                base_name = os.path.splitext(file)[0]
                image_map[base_name] = os.path.join(root, file)
                
    print(f"✅ 주소록 완성! 총 {len(image_map)}개의 이미지 위치를 파악했습니다.")
    return image_map

def process_new_data_fast():
    # 1. 준비 단계
    target_list = get_allowed_drug_list()
    if not target_list: return
    
    # 여기서 이미지 주소록을 먼저 만듭니다 (시간 단축의 핵심)
    img_map = build_image_map(new_img_root)

    os.makedirs(save_root, exist_ok=True)
    print(f"🚀 [초고속 모드] NEW 데이터 크롭 시작 (대상: 56종)")
    
    stats = {"success": 0, "skip_filter": 0, "skip_no_img": 0}

    # 2. 라벨 탐색
    for root, dirs, files in os.walk(new_lbl_root):
        folder_name = os.path.basename(root)
        
        # 약물 코드 추출
        drug_code = None
        if folder_name.startswith("K-") and len(folder_name) > 5:
            drug_code = folder_name
        
        if not drug_code: continue

        # 족보 필터링
        if drug_code not in target_list:
            stats["skip_filter"] += len([f for f in files if f.endswith(".json")])
            continue 

        # 3. 고속 처리 시작
        json_files = [f for f in files if f.endswith(".json")]
        
        # tqdm으로 진행률 표시
        for j_file in tqdm(json_files, desc=f"{drug_code}", leave=False):
            try:
                # JSON 로드
                json_path = os.path.join(root, j_file)
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                if not data['annotations']: continue
                bbox = data['annotations'][0]['bbox']
                x, y, w, h = map(int, bbox)

                # 🔥 [핵심] 주소록에서 바로 조회 (검색 시간 0초)
                base_name = os.path.splitext(j_file)[0]
                img_path = img_map.get(base_name) # 여기서 바로 찾음!
                
                if not img_path:
                    stats["skip_no_img"] += 1; continue
                
                img = cv2.imread(img_path)
                if img is None: continue
                img_h, img_w, _ = img.shape

                # 좌표 계산 (안전 크롭)
                center_x, center_y = x + w/2, y + h/2
                new_w, new_h = w * SCALE_FACTOR, h * SCALE_FACTOR
                new_x, new_y = center_x - new_w/2, center_y - new_h/2
                
                x1 = int(max(0, new_x))
                y1 = int(max(0, new_y))
                x2 = int(min(img_w, new_x + new_w))
                y2 = int(min(img_h, new_y + new_h))
                
                if x2 <= x1 or y2 <= y1: continue

                crop = img[y1:y2, x1:x2]
                
                # 저장
                save_dir = os.path.join(save_root, drug_code)
                os.makedirs(save_dir, exist_ok=True)
                save_name = f"NEW_{base_name}_{drug_code}.jpg"
                cv2.imwrite(os.path.join(save_dir, save_name), crop)
                stats["success"] += 1

            except:
                continue

    print("=" * 50)
    print(f"✅ 초고속 작업 완료")
    print(f"   - 성공: {stats['success']}장")
    print(f"   - 필터링(버림): {stats['skip_filter']}장")
    print(f"   - 이미지 못찾음: {stats['skip_no_img']}장")
    print(f"📂 저장 위치: {save_root}")
    print("=" * 50)

# 실행
process_new_data_fast()

📋 족보(56종) 확인 중...
🗺️ 이미지 위치 주소록(Indexing) 만드는 중... (잠시만 기다려주세요)
✅ 주소록 완성! 총 16012개의 이미지 위치를 파악했습니다.
🚀 [초고속 모드] NEW 데이터 크롭 시작 (대상: 56종)


✅ 초고속 작업 완료
   - 성공: 23083장
   - 필터링(버림): 22921장
   - 이미지 못찾음: 17장
📂 저장 위치: C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY1.2


# 크롭 이미지 갯수 파악(원본 이미지 + 크롭 이미지)

In [5]:
import os
import pandas as pd
from collections import defaultdict

# ==========================================
# 📂 크롭 이미지 경로만 설정
# ==========================================
crop_old_path = r"C:\Users\home\Desktop\AI_study\processed_crops_old_ONLY1.2"
crop_new_path = r"C:\Users\home\Desktop\AI_study\processed_crops_NEW_ONLY1.2"

def count_images_only():
    print("🔎 폴더별 크롭 이미지 개수를 집계 중입니다...")
    
    # stats[약ID] = [OLD개수, NEW개수]
    stats = defaultdict(lambda: [0, 0])

    # 1. OLD 폴더 집계
    if os.path.exists(crop_old_path):
        for root, dirs, files in os.walk(crop_old_path):
            # 폴더명에서 약 ID(K-로 시작하는 이름) 추출
            folder_name = os.path.basename(root)
            if folder_name.startswith("K-"):
                img_count = len([f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
                stats[folder_name][0] += img_count

    # 2. NEW 폴더 집계
    if os.path.exists(crop_new_path):
        for root, dirs, files in os.walk(crop_new_path):
            folder_name = os.path.basename(root)
            if folder_name.startswith("K-"):
                img_count = len([f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
                stats[folder_name][1] += img_count

    # 3. 데이터 정리
    data = []
    for drug_id, counts in stats.items():
        old_c, new_c = counts
        data.append({
            'Drug_ID': drug_id,
            'OLD_Crop': old_c,
            'NEW_Crop': new_c,
            'Total': old_c + new_c
        })

    df = pd.DataFrame(data)
    
    if df.empty:
        print("❌ 이미지를 찾지 못했습니다. 경로를 다시 확인해주세요.")
        return

    # 개수가 적은 순서대로 정렬 (증강 필요 대상 파악용)
    df = df.sort_values(by='Total', ascending=True).reset_index(drop=True)

    # 4. 출력
    print("\n" + "="*60)
    print(f"{'Drug ID':<15} | {'OLD':>8} | {'NEW':>8} | {'Total':>8}")
    print("-" * 60)
    
    for _, row in df.iterrows():
        print(f"{row['Drug_ID']:<15} | {row['OLD_Crop']:>8} | {row['NEW_Crop']:>8} | {row['Total']:>8}")

    print("=" * 60)
    print(f"✅ 총 {len(df)}종의 약물 데이터 집계 완료")
    
    # 엑셀 저장 (필요시 확인용)
    df.to_csv(r"C:\Users\home\Desktop\AI_study\Crop_Count_Final.csv", index=False, encoding='utf-8-sig')
    
    return df

# 실행
count_df = count_images_only()

🔎 폴더별 크롭 이미지 개수를 집계 중입니다...

Drug ID         |      OLD |      NEW |    Total
------------------------------------------------------------
K-016688        |        8 |      156 |      164
K-031863        |       12 |      156 |      168
K-032310        |       14 |      155 |      169
K-027993        |        3 |      168 |      171
K-033880        |       16 |      156 |      172
K-018357        |       12 |      160 |      172
K-018147        |       15 |      157 |      172
K-020014        |       16 |      160 |      176
K-041768        |       16 |      162 |      178
K-019232        |       17 |      162 |      179
K-022074        |       16 |      171 |      187
K-038162        |       18 |      175 |      193
K-021325        |       22 |      176 |      198
K-033208        |        3 |      232 |      235
K-029345        |        6 |      231 |      237
K-033009        |        3 |      235 |      238
K-029451        |        3 |      235 |      238
K-024850        |        3 |